# Election Header Extraction

This notebook OCRs one representative image per province code using `constituency_<province_code>_1.png`, builds a province mapping, and then generates `doc_id`, `province_name`, and `constituency` for every base document in `data/images`.

In [ ]:
import csv
import difflib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

try:
    import requests
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install election notebook dependencies first, for example by running `uv sync` in notebooks/election."
    ) from exc

OCR_URL = "https://api.opentyphoon.ai/v1/ocr"
OCR_MODEL = "typhoon-ocr"
OCR_TASK_TYPE = "default"
OCR_MAX_TOKENS = 16384
OCR_TEMPERATURE = 0.1
OCR_TOP_P = 0.6
OCR_REPETITION_PENALTY = 1.2
ONLY_PROVINCE_CODES = []
STOP_ON_ERROR = True
OUTPUT_DIRNAME = "outputs"
OUTPUT_CSV_NAME = "header_extract.csv"
PROGRESS_LOG_NAME = "header_extract_progress.jsonl"
DOC_RE = re.compile(
    r"^(?P<doc_type>constituency|party_list)_(?P<province_code>\d+)_(?P<constituency>\d+)\.png$"
)
REPRESENTATIVE_RE = re.compile(r"^constituency_(?P<province_code>\d+)_1\.png$")
FIELDNAMES = ["doc_id", "province_name", "constituency"]
VALID_THAI_PROVINCES = [
    "กรุงเทพมหานคร",
    "กระบี่",
    "กาญจนบุรี",
    "กาฬสินธุ์",
    "กำแพงเพชร",
    "ขอนแก่น",
    "จันทบุรี",
    "ฉะเชิงเทรา",
    "ชลบุรี",
    "ชัยนาท",
    "ชัยภูมิ",
    "ชุมพร",
    "เชียงราย",
    "เชียงใหม่",
    "ตรัง",
    "ตราด",
    "ตาก",
    "นครนายก",
    "นครปฐม",
    "นครพนม",
    "นครราชสีมา",
    "นครศรีธรรมราช",
    "นครสวรรค์",
    "นนทบุรี",
    "นราธิวาส",
    "น่าน",
    "บึงกาฬ",
    "บุรีรัมย์",
    "ปทุมธานี",
    "ประจวบคีรีขันธ์",
    "ปราจีนบุรี",
    "ปัตตานี",
    "พระนครศรีอยุธยา",
    "พะเยา",
    "พังงา",
    "พัทลุง",
    "พิจิตร",
    "พิษณุโลก",
    "เพชรบุรี",
    "เพชรบูรณ์",
    "แพร่",
    "ภูเก็ต",
    "มหาสารคาม",
    "มุกดาหาร",
    "แม่ฮ่องสอน",
    "ยโสธร",
    "ยะลา",
    "ร้อยเอ็ด",
    "ระนอง",
    "ระยอง",
    "ราชบุรี",
    "ลพบุรี",
    "ลำปาง",
    "ลำพูน",
    "เลย",
    "ศรีสะเกษ",
    "สกลนคร",
    "สงขลา",
    "สตูล",
    "สมุทรปราการ",
    "สมุทรสงคราม",
    "สมุทรสาคร",
    "สระแก้ว",
    "สระบุรี",
    "สิงห์บุรี",
    "สุโขทัย",
    "สุพรรณบุรี",
    "สุราษฎร์ธานี",
    "สุรินทร์",
    "หนองคาย",
    "หนองบัวลำภู",
    "อ่างทอง",
    "อำนาจเจริญ",
    "อุดรธานี",
    "อุตรดิตถ์",
    "อุทัยธานี",
    "อุบลราชธานี",
]
PROVINCE_NAME_MAP = {
    re.sub(r"[^ก-๙a-z0-9]", "", name.lower()): name for name in VALID_THAI_PROVINCES
}


def guess_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "notebooks" / "election",
        Path.cwd().parent,
        Path.cwd().parent / "notebooks" / "election",
    ]
    for candidate in candidates:
        if (candidate / "data" / "images").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the election notebook directory from the current working directory."
    )


BASE_DIR = guess_base_dir()
ENV_PATH = BASE_DIR / ".env"
IMAGE_DIR = BASE_DIR / "data" / "images"
OUTPUT_DIR = BASE_DIR / OUTPUT_DIRNAME
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV_NAME
PROGRESS_LOG_PATH = OUTPUT_DIR / PROGRESS_LOG_NAME


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(ENV_PATH)
OPENTYPHOON_API_KEY = os.environ.get("OPENTYPHOON_API_KEY", "").strip()

print("BASE_DIR:", BASE_DIR)
print("IMAGE_DIR:", IMAGE_DIR)
print("OUTPUT_CSV_PATH:", OUTPUT_CSV_PATH)
print("PROGRESS_LOG_PATH:", PROGRESS_LOG_PATH)
print("OPENTYPHOON_API_KEY loaded:", bool(OPENTYPHOON_API_KEY))

In [ ]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_jsonl(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")


def load_progress_entries(path: Path) -> list[dict[str, object]]:
    if not path.exists():
        return []
    entries = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            entries.append(json.loads(line))
    return entries


def load_successful_province_map(path: Path) -> dict[int, str]:
    province_map = {}
    for entry in load_progress_entries(path):
        if (
            entry.get("status") == "success"
            and "province_code" in entry
            and "province_name" in entry
        ):
            province_map[int(entry["province_code"])] = str(entry["province_name"])
    return province_map


def write_csv_rows(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(rows)


def load_saved_rows(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def discover_base_documents(image_dir: Path) -> list[dict[str, object]]:
    docs = []
    for image_path in sorted(image_dir.glob("*.png")):
        match = DOC_RE.match(image_path.name)
        if not match:
            continue
        docs.append(
            {
                "doc_id": image_path.stem,
                "doc_type": match.group("doc_type"),
                "province_code": int(match.group("province_code")),
                "constituency": int(match.group("constituency")),
                "image_path": image_path,
            }
        )
    return docs


def discover_representative_documents(image_dir: Path) -> list[dict[str, object]]:
    representatives = []
    for image_path in sorted(image_dir.glob("constituency_*_1.png")):
        match = REPRESENTATIVE_RE.match(image_path.name)
        if not match:
            continue
        province_code = int(match.group("province_code"))
        representatives.append(
            {
                "province_code": province_code,
                "doc_id": image_path.stem,
                "image_path": image_path,
            }
        )
    return representatives


ALL_DOCS = discover_base_documents(IMAGE_DIR)
REPRESENTATIVE_DOCS = discover_representative_documents(IMAGE_DIR)
SELECTED_REPRESENTATIVES = [
    doc
    for doc in REPRESENTATIVE_DOCS
    if not ONLY_PROVINCE_CODES or doc["province_code"] in ONLY_PROVINCE_CODES
]
PROVINCE_MAP_FROM_LOG = load_successful_province_map(PROGRESS_LOG_PATH)
PENDING_REPRESENTATIVES = [
    doc
    for doc in SELECTED_REPRESENTATIVES
    if doc["province_code"] not in PROVINCE_MAP_FROM_LOG
]

print(
    {
        "all_base_docs": len(ALL_DOCS),
        "representative_docs": len(REPRESENTATIVE_DOCS),
        "selected_representatives": len(SELECTED_REPRESENTATIVES),
        "completed_province_codes": len(PROVINCE_MAP_FROM_LOG),
        "pending_representatives": len(PENDING_REPRESENTATIVES),
    }
)

In [ ]:
def extract_text_from_image(
    image_path: Path,
    api_key: str,
    model: str = OCR_MODEL,
    task_type: str = OCR_TASK_TYPE,
    max_tokens: int = OCR_MAX_TOKENS,
    temperature: float = OCR_TEMPERATURE,
    top_p: float = OCR_TOP_P,
    repetition_penalty: float = OCR_REPETITION_PENALTY,
    pages: list[int] | None = None,
) -> str:
    if not api_key:
        raise ValueError(
            "Set OPENTYPHOON_API_KEY in notebooks/election/.env before calling the OCR API."
        )

    with image_path.open("rb") as file_handle:
        files = {"file": file_handle}
        data = {
            "model": model,
            "task_type": task_type,
            "max_tokens": str(max_tokens),
            "temperature": str(temperature),
            "top_p": str(top_p),
            "repetition_penalty": str(repetition_penalty),
        }
        if pages:
            data["pages"] = json.dumps(pages)

        headers = {"Authorization": f"Bearer {api_key}"}
        response = requests.post(
            OCR_URL, files=files, data=data, headers=headers, timeout=180
        )

    if response.status_code != 200:
        raise RuntimeError(
            f"OCR request failed with status {response.status_code}: {response.text}"
        )

    result = response.json()
    extracted_texts = []
    for page_result in result.get("results", []):
        if page_result.get("success") and page_result.get("message"):
            content = page_result["message"]["choices"][0]["message"]["content"]
            try:
                parsed_content = json.loads(content)
                text = parsed_content.get("natural_text", content)
            except json.JSONDecodeError:
                text = content
            extracted_texts.append(text)
        else:
            raise RuntimeError(
                f"OCR failed for {page_result.get('filename', image_path.name)}: {page_result.get('error', 'Unknown error')}"
            )

    if not extracted_texts:
        raise RuntimeError(f"OCR returned no text for {image_path.name}")
    return "\n".join(text.strip() for text in extracted_texts if text and text.strip())

In [ ]:
def normalize_for_lookup(text: str) -> str:
    return re.sub(r"[^ก-๙a-z0-9]", "", text.lower())


def strip_ocr_line(line: str) -> str:
    line = re.sub(r"<[^>]+>", " ", line)
    line = re.sub(r"\s+", " ", line).strip()
    return line.strip("-:|[]{}()")


def iter_candidate_lines(ocr_text: str, max_lines: int = 30) -> list[str]:
    candidates = []
    for raw_line in ocr_text.splitlines():
        line = strip_ocr_line(raw_line)
        if not line or line == "---":
            continue
        if ":" in line and any(
            token in line.lower()
            for token in [
                "primary_language",
                "is_rotation_valid",
                "rotation_correction",
                "is_table",
                "is_diagram",
            ]
        ):
            continue
        if line.startswith("<") or line.startswith("{"):
            continue
        candidates.append(line)
        if len(candidates) >= max_lines:
            break
    return candidates


def extract_province_name(ocr_text: str) -> str:
    candidate_lines = iter_candidate_lines(ocr_text)
    top_text = "\n".join(candidate_lines)

    for line in candidate_lines:
        if line in VALID_THAI_PROVINCES:
            return line

    for province_name in VALID_THAI_PROVINCES:
        if province_name in top_text:
            return province_name

    normalized_candidates = []
    for line in candidate_lines:
        normalized = normalize_for_lookup(line)
        if not normalized:
            continue
        normalized_candidates.append(normalized)
        if normalized in PROVINCE_NAME_MAP:
            return PROVINCE_NAME_MAP[normalized]

    for normalized_line in normalized_candidates:
        matches = difflib.get_close_matches(
            normalized_line, PROVINCE_NAME_MAP.keys(), n=1, cutoff=0.72
        )
        if matches:
            return PROVINCE_NAME_MAP[matches[0]]

    raise ValueError(
        f"Could not extract a valid Thai province name from OCR text. Top lines: {candidate_lines[:10]}"
    )


def validate_row(row: dict[str, object]) -> None:
    if not str(row["doc_id"]).strip():
        raise ValueError("doc_id must not be empty")
    if row["province_name"] not in VALID_THAI_PROVINCES:
        raise ValueError(f"Invalid province_name: {row['province_name']}")
    if int(row["constituency"]) <= 0:
        raise ValueError(f"constituency must be positive: {row['constituency']}")


def build_header_row(
    doc: dict[str, object], province_map: dict[int, str]
) -> dict[str, object]:
    province_name = province_map[int(doc["province_code"])]
    row = {
        "doc_id": doc["doc_id"],
        "province_name": province_name,
        "constituency": int(doc["constituency"]),
    }
    validate_row(row)
    return row

In [ ]:
assert len(ALL_DOCS) == 300, f"Expected 300 base docs, found {len(ALL_DOCS)}"
assert all("_page" not in doc["image_path"].name for doc in ALL_DOCS)
assert any(doc["doc_type"] == "constituency" for doc in ALL_DOCS)
assert any(doc["doc_type"] == "party_list" for doc in ALL_DOCS)
assert len(REPRESENTATIVE_DOCS) == len(
    {doc["province_code"] for doc in ALL_DOCS if doc["doc_type"] == "constituency"}
)
assert {doc["province_code"] for doc in ALL_DOCS}.issubset(
    {doc["province_code"] for doc in REPRESENTATIVE_DOCS}
)

print("Discovery checks passed.")
print("First 5 docs:", [doc["doc_id"] for doc in ALL_DOCS[:5]])
print(
    "Representative province codes:",
    [doc["province_code"] for doc in REPRESENTATIVE_DOCS[:10]],
)

In [ ]:
SMOKE_PROVINCE_CODES = [10, 11]

if not OPENTYPHOON_API_KEY:
    print("Smoke test skipped because OPENTYPHOON_API_KEY is not set.")
else:
    smoke_representatives = [
        doc
        for doc in REPRESENTATIVE_DOCS
        if doc["province_code"] in SMOKE_PROVINCE_CODES
    ]
    smoke_map = {}
    for doc in smoke_representatives:
        ocr_text = extract_text_from_image(doc["image_path"], OPENTYPHOON_API_KEY)
        smoke_map[int(doc["province_code"])] = extract_province_name(ocr_text)

    smoke_rows = []
    for doc_id in ["constituency_10_1", "constituency_10_2", "party_list_10_2"]:
        doc = next(doc for doc in ALL_DOCS if doc["doc_id"] == doc_id)
        smoke_rows.append(build_header_row(doc, smoke_map))

    print({"province_map": smoke_map, "rows": smoke_rows})

In [ ]:
def process_one_representative(doc: dict[str, object]) -> tuple[int, str]:
    ocr_text = extract_text_from_image(doc["image_path"], OPENTYPHOON_API_KEY)
    province_name = extract_province_name(ocr_text)
    append_jsonl(
        PROGRESS_LOG_PATH,
        {
            "timestamp": utc_now_iso(),
            "status": "success",
            "province_code": int(doc["province_code"]),
            "doc_id": doc["doc_id"],
            "image_path": str(doc["image_path"]),
            "province_name": province_name,
        },
    )
    return int(doc["province_code"]), province_name


if not OPENTYPHOON_API_KEY:
    raise ValueError(
        "Set OPENTYPHOON_API_KEY in notebooks/election/.env before running the representative OCR loop."
    )

province_map = dict(PROVINCE_MAP_FROM_LOG)
run_summary = []
for index, doc in enumerate(PENDING_REPRESENTATIVES, start=1):
    print(
        f"[{index}/{len(PENDING_REPRESENTATIVES)}] Processing province_code={doc['province_code']} via {doc['doc_id']}"
    )
    try:
        province_code, province_name = process_one_representative(doc)
        province_map[province_code] = province_name
        run_summary.append(
            {"province_code": province_code, "province_name": province_name}
        )
        print({"province_code": province_code, "province_name": province_name})
    except Exception as exc:
        append_jsonl(
            PROGRESS_LOG_PATH,
            {
                "timestamp": utc_now_iso(),
                "status": "failed",
                "province_code": int(doc["province_code"]),
                "doc_id": doc["doc_id"],
                "image_path": str(doc["image_path"]),
                "error": str(exc),
            },
        )
        print(
            {
                "province_code": doc["province_code"],
                "status": "failed",
                "error": str(exc),
            }
        )
        if STOP_ON_ERROR:
            raise

missing_codes = sorted({doc["province_code"] for doc in ALL_DOCS} - set(province_map))
if missing_codes:
    raise ValueError(f"Missing province mappings for province codes: {missing_codes}")

final_rows = [build_header_row(doc, province_map) for doc in ALL_DOCS]
write_csv_rows(OUTPUT_CSV_PATH, final_rows)

print(
    {
        "processed_representatives_now": len(run_summary),
        "province_codes_available": len(province_map),
        "written_rows": len(final_rows),
        "output_csv": str(OUTPUT_CSV_PATH),
    }
)

In [ ]:
saved_rows = load_saved_rows(OUTPUT_CSV_PATH)
province_codes_in_docs = {doc["province_code"] for doc in ALL_DOCS}
province_map_from_log = load_successful_province_map(PROGRESS_LOG_PATH)

assert len(saved_rows) == len(ALL_DOCS)
assert all(row["doc_id"].strip() for row in saved_rows)
assert all(row["province_name"] in VALID_THAI_PROVINCES for row in saved_rows)
assert all(int(row["constituency"]) > 0 for row in saved_rows)
assert len({row["doc_id"] for row in saved_rows}) == len(saved_rows)
assert province_codes_in_docs.issubset(set(province_map_from_log))

lookup = {row["doc_id"]: row for row in saved_rows}
if "constituency_10_1" in lookup:
    assert lookup["constituency_10_1"]["province_name"] == "กรุงเทพมหานคร"
    assert int(lookup["constituency_10_1"]["constituency"]) == 1
if "constituency_10_2" in lookup:
    assert lookup["constituency_10_2"]["province_name"] == "กรุงเทพมหานคร"
    assert int(lookup["constituency_10_2"]["constituency"]) == 2
if "party_list_10_2" in lookup:
    assert lookup["party_list_10_2"]["province_name"] == "กรุงเทพมหานคร"
    assert int(lookup["party_list_10_2"]["constituency"]) == 2

print(
    {
        "saved_rows": len(saved_rows),
        "province_codes_mapped": len(province_map_from_log),
        "sample_rows": [
            lookup[key]
            for key in ["constituency_10_1", "constituency_10_2", "party_list_10_2"]
            if key in lookup
        ],
    }
)